<a href="https://colab.research.google.com/github/ishuu9837/Fine-tuning-Llama-3.2/blob/main/Fine_Tuning_Llama_3_2_for_Domain_Specific_(Finance)_Question_Answering_using_LoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -q "transformers>=4.56,<5" "trl==0.23.1" "peft>=0.17,<0.19" "accelerate>=1.4,<2" "bitsandbytes>=0.46,<0.49" "datasets>=3,<5" evaluate rouge_score sacrebleu

In [ ]:
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get('collab'))

In [ ]:
# Restart the Colab runtime after installing dependencies, then run from the login cell.
import torch
from transformers import set_seed
SEED = 42
set_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU in Colab.")
use_bf16 = torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16
print(f"GPU: {torch.cuda.get_device_name(0)}; compute dtype: {compute_dtype}")

In [ ]:
from datasets import load_dataset

dataset = load_dataset("gbharti/finance-alpaca", split="train")
# Keep each normalized question/input in just one partition, even if answers differ.
seen = set()
def unique_question(example):
    if not example["instruction"].strip() or not example["output"].strip():
        return False
    key = (" ".join(example["instruction"].lower().split()),
           " ".join(example["input"].lower().split()))
    if key in seen:
        return False
    seen.add(key)
    return True

dataset = dataset.filter(unique_question, load_from_cache_file=False)
print(f"Unique nonempty questions: {len(dataset)}")

In [ ]:
from transformers import AutoTokenizer

model_id = "meta-llama/Llama-3.2-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

def format_example(example):
    question = example["instruction"].strip()
    if example["input"].strip():
        question += "\n\n" + example["input"].strip()
    return {
        "prompt": [{"role": "user", "content": question}],
        "completion": [{"role": "assistant", "content": example["output"].strip()}],
    }

dataset = dataset.map(format_example)

In [ ]:
# Final test data is never used to select checkpoints.
split = dataset.train_test_split(test_size=0.2, seed=SEED)
heldout = split["test"].train_test_split(test_size=0.5, seed=SEED)
train_dataset = split["train"]
eval_dataset = heldout["train"]
test_dataset = heldout["test"]

In [ ]:
train_dataset = train_dataset.shuffle(seed=SEED).select(range(min(3000, len(train_dataset))))
eval_dataset = eval_dataset.shuffle(seed=SEED).select(range(min(200, len(eval_dataset))))
raw_test = test_dataset.shuffle(seed=SEED).select(range(min(50, len(test_dataset))))
print(f"Train: {len(train_dataset)}; validation: {len(eval_dataset)}; final test: {len(raw_test)}")

In [ ]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={"": torch.cuda.current_device()},
)
model.config.use_cache = False
print("Model loaded successfully!")

In [ ]:
print(model)

In [ ]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from trl import SFTConfig

sft_config = SFTConfig(
    output_dir="./llama3-finance-lora",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=11,
    logging_steps=10,
    eval_strategy="steps", eval_steps=50,
    save_strategy="steps", save_steps=50, save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss", greater_is_better=False,
    bf16=use_bf16, fp16=not use_bf16,
    max_length=512, packing=False,
    completion_only_loss=True,
    eos_token="<|eot_id|>",
    seed=SEED, data_seed=SEED, report_to="none",
)

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model, args=sft_config,
    train_dataset=train_dataset.select_columns(["prompt", "completion"]),
    eval_dataset=eval_dataset.select_columns(["prompt", "completion"]),
    processing_class=tokenizer,
)
trainer.train()

In [ ]:
trainer.save_model("./llama3-finance-lora-final")
tokenizer.save_pretrained("./llama3-finance-lora-final")

In [ ]:
import math
from collections import Counter
from contextlib import nullcontext

# All final metrics use the same held-out examples, prompt, and decoding settings.
model = trainer.model
model.eval()
model.config.use_cache = True
MAX_TEST_LENGTH = 1024
MAX_NEW_TOKENS = 256

def normalize_answer(text):
    # Preserve punctuation, signs and decimal points: finance numbers must stay distinct.
    return " ".join(text.lower().split())

def answer_scores(predictions, references):
    if not references or len(predictions) != len(references):
        raise ValueError("Need equal, nonempty prediction/reference lists")
    exact, f1 = 0, 0.0
    for prediction, reference in zip(predictions, references):
        a, b = normalize_answer(prediction), normalize_answer(reference)
        exact += a == b
        at, bt = a.split(), b.split()
        overlap = sum((Counter(at) & Counter(bt)).values())
        f1 += 2 * overlap / (len(at) + len(bt)) if at or bt else 1.0
    return {"exact_match_accuracy_pct": 100 * exact / len(references),
            "answer_token_f1_pct": 100 * f1 / len(references)}

def generate_answer(example):
    prompt = tokenizer.apply_chat_template(example["prompt"], tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, add_special_tokens=False, return_tensors="pt").to(model.device)
    with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=compute_dtype):
        outputs = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS,
                                 do_sample=False, pad_token_id=tokenizer.pad_token_id,
                                 use_cache=True)
    return tokenizer.decode(outputs[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def answer_likelihood(examples):
    loss_sum, count, correct, skipped, truncated = 0.0, 0, 0, 0, 0
    for example in examples:
        prompt = tokenizer.apply_chat_template(example["prompt"], tokenize=False, add_generation_prompt=True)
        full = tokenizer.apply_chat_template(example["prompt"] + example["completion"], tokenize=False)
        prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
        full_ids = tokenizer(full, add_special_tokens=False)["input_ids"]
        if full_ids[:len(prompt_ids)] != prompt_ids:
            raise ValueError("Chat template prompt is not a prefix of the full conversation")
        if len(full_ids) > MAX_TEST_LENGTH:
            truncated += 1
        ids = full_ids[:MAX_TEST_LENGTH]
        if len(prompt_ids) >= len(ids):
            skipped += 1
            continue
        inputs = torch.tensor([ids], device=model.device)
        labels = inputs.clone()
        labels[:, :len(prompt_ids)] = -100
        with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=compute_dtype):
            out = model(input_ids=inputs, labels=labels)
        targets = labels[:, 1:]
        mask = targets != -100
        tokens = int(mask.sum().item())
        loss_sum += out.loss.item() * tokens
        correct += int(((out.logits[:, :-1].argmax(-1) == targets) & mask).sum().item())
        count += tokens
    if not count:
        raise ValueError("No answer tokens fit in the test context window")
    loss = loss_sum / count
    return {"answer_loss": loss, "answer_perplexity": math.exp(loss),
            "next_token_accuracy_pct": 100 * correct / count,
            "evaluated_answer_tokens": count, "skipped_examples": skipped,
            "truncated_examples": truncated}

In [ ]:
import json
import evaluate
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")
references = [e["output"].strip() for e in raw_test]
results, answers = {}, {}
for name in ("base", "fine_tuned"):
    # Reuse the same quantized weights to avoid loading two 3B models onto a T4.
    context = model.disable_adapter() if name == "base" else nullcontext()
    with context:
        predictions = [generate_answer(e) for e in raw_test]
        scores = answer_scores(predictions, references)
        scores.update(answer_likelihood(raw_test))
    scores["bleu_pct"] = 100 * bleu.compute(predictions=predictions, references=[[r] for r in references])["bleu"]
    scores.update({f"{k}_pct": 100 * v for k, v in rouge.compute(predictions=predictions, references=references).items()})
    results[name], answers[name] = scores, predictions
    print(name, json.dumps(scores, indent=2))

In [ ]:
import pandas as pd
from IPython.display import display
comparison = pd.DataFrame(results)
comparison["change"] = comparison["fine_tuned"] - comparison["base"]
display(comparison.round(4))
print("Accuracy/F1/ROUGE/BLEU changes are percentage points. Lower loss/perplexity is better.")
print("Next-token accuracy uses the correct preceding answer tokens; it is not factual QA accuracy.")
print("Exact match is strict answer equality after lowercasing and whitespace normalization.")

In [ ]:
# Save measured scores, provenance, and generated answers for review.
from pathlib import Path
import importlib.metadata
output = Path("evaluation_results")
output.mkdir(exist_ok=True)
report = {
    "model_id": model_id, "dataset": "gbharti/finance-alpaca", "seed": SEED,
    "train_examples": len(train_dataset), "validation_examples": len(eval_dataset),
    "test_examples": len(raw_test), "max_test_length": MAX_TEST_LENGTH,
    "max_new_tokens": MAX_NEW_TOKENS,
    "versions": {p: importlib.metadata.version(p) for p in ["torch", "transformers", "trl", "peft", "datasets"]},
    "results": results,
    "examples": [{"instruction": e["instruction"], "input": e["input"],
                  "reference": references[i], "base": answers["base"][i],
                  "fine_tuned": answers["fine_tuned"][i]} for i, e in enumerate(raw_test)],
}
(output / "metrics.json").write_text(json.dumps(report, indent=2))
comparison.to_csv(output / "comparison.csv")
print(f"Saved results to {output.resolve()}")

## Reading the numbers

Exact-match accuracy and next-token accuracy are different measurements. Token F1 and ROUGE measure answer overlap, not financial correctness. A 50-question sample is exploratory; increase the test size for stronger evidence. The test set is separate from checkpoint validation. Inspect skipped/truncated counts alongside likelihood metrics.

In [ ]:
# For a larger final evaluation, increase raw_test above and rerun the evaluation cells.

In [ ]:
for i in range(min(3, len(raw_test))):
    print(f"Question: {raw_test[i]['instruction']}")
    print(f"Base: {answers['base'][i]}")
    print(f"Fine-tuned: {answers['fine_tuned'][i]}")
    print(f"Reference: {references[i]}\n")

In [ ]:
!zip -rq llama3-finance-results.zip ./llama3-finance-lora-final ./evaluation_results
from google.colab import files
files.download("llama3-finance-results.zip")